# Day 2 — Step 4: 시각화 [도전 과제 — 문제3]

03_moving_average에서 저장한 최종 데이터를 불러와  
종가(close)와 5일 이동평균선(MA5)을 차트로 시각화합니다.

---

## matplotlib이란?

파이썬으로 그래프를 그릴 때 가장 많이 쓰는 라이브러리입니다.  
1주차에서는 plotly(인터랙티브 차트)를 썼는데, 이번에는 matplotlib을 써봤습니다.

| | matplotlib | plotly (1주차) |
|--|-----------|----------------|
| 인터랙션 | ❌ 정적 이미지 | ✅ 마우스로 조작 가능 |
| 이미지 저장 | ✅ 쉬움 (.png) | 별도 설정 필요 |
| 난이도 | 상대적으로 단순 | 옵션이 다양해서 복잡 |

---

## 서브플롯(subplot)이란?

하나의 그림 안에 여러 개의 차트를 배치하는 것입니다.  
3종목을 한 화면에 나란히 보여주기 위해 1행 3열 서브플롯을 만들었습니다.

```
fig (전체 그림)
 ├── axes[0]  ← KRW-BTC 차트
 ├── axes[1]  ← KRW-ETH 차트
 └── axes[2]  ← KRW-SOL 차트
```

---

## 차트를 보는 방법

| 상황 | 의미 |
|------|------|
| 종가(검정선)가 MA5(빨간선) **위** | 최근 5일 평균보다 높다 → 단기 상승 중 |
| 종가(검정선)가 MA5(빨간선) **아래** | 최근 5일 평균보다 낮다 → 단기 하락 중 |
| MA5선이 **우상향** | 5일 평균이 올라가고 있다 → 상승 추세 |
| MA5선이 **우하향** | 5일 평균이 내려가고 있다 → 하락 추세 |

---
## 0. 라이브러리 불러오기

| 라이브러리 | 비유 | 우리가 쓰는 이유 |
|-----------|------|------------------|
| matplotlib.pyplot | 그림판 | 선 그래프, 차트를 그려줌 |
| matplotlib.dates | 달력 눈금 | x축에 날짜를 보기 좋게 표시해줌 |
| matplotlib.ticker | 숫자 자 | y축 숫자에 쉼표를 넣어줌 (89,250,000처럼) |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt          # 차트(그래프) 그리기
import matplotlib.dates as mdates        # x축 날짜 포맷 설정
import matplotlib.ticker as mticker      # y축 숫자 포맷 설정
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 불러오기 완료!")

---
## 1. 데이터 불러오기

In [ ]:
df = pd.read_csv('ohlcv_final.csv', parse_dates=['date'])

print(f"데이터 로드 완료: {len(df)}행")
print(df.head())

---
## 2. [도전 과제 — 문제3] 종가와 5일 이동평균선 시각화

In [ ]:
def prepare_plot_data(df):
    """
    시각화 전에 종목별로 데이터를 분리합니다.

    반환값(Return):
        {"KRW-BTC": DataFrame, "KRW-ETH": DataFrame, ...} 형태의 딕셔너리
    """
    df = df.copy()

    # date 컬럼이 datetime 타입인지 확인 (mdates는 datetime 타입 필요)
    df['date'] = pd.to_datetime(df['date'])

    # 종목별로 데이터 분리해서 딕셔너리에 저장
    # .unique() : 중복 없이 유일한 ticker 목록만 반환
    ticker_data = {}
    for ticker in df['ticker'].unique():
        ticker_df = df[df['ticker'] == ticker].sort_values('date').copy()
        ticker_data[ticker] = ticker_df

    return ticker_data

In [ ]:
def plot_close_and_ma5(ticker_data):
    """
    3종목의 종가(close)와 5일 이동평균(ma5)을 서브플롯으로 시각화합니다.

    차트 구성:
        - 1행 3열 서브플롯 (종목당 1칸)
        - 검정 실선 : 종가(close)
        - 빨간 점선 : 5일 이동평균(ma5)
        - x축 : 날짜 (MM-DD 형식)
        - y축 : 가격 (원 단위)

    매개변수(Parameter):
        ticker_data : prepare_plot_data()에서 반환된 딕셔너리
    """
    # ── 1행 3열 서브플롯 생성 ─────────────────────────────────────
    # plt.subplots(행 수, 열 수)
    # figsize=(18, 5) : 전체 그림 크기 (가로 18인치, 세로 5인치)
    # fig   : 전체 그림 객체 (캔버스)
    # axes  : 각 차트 칸 배열 [axes[0], axes[1], axes[2]]
    fig, axes = plt.subplots(nrows=1, ncols=len(ticker_data), figsize=(18, 5))

    # ── 종목별로 차트 그리기 ──────────────────────────────────────
    # enumerate() : 인덱스(i)와 값(ticker)을 동시에 반환
    # i=0 → axes[0] (첫 번째 칸), i=1 → axes[1] (두 번째 칸) ...
    for i, ticker in enumerate(ticker_data):
        ax  = axes[i]               # i번째 차트 칸
        tdf = ticker_data[ticker]   # 해당 종목 데이터

        # ── 종가 선 그리기 ────────────────────────────────────────
        # ax.plot(x값, y값, 스타일 옵션)
        # color='black'     : 선 색상 → 검정
        # linewidth=1.5     : 선 굵기 (숫자 클수록 굵음)
        # label='종가...'   : 범례에 표시될 이름
        ax.plot(tdf['date'], tdf['close'],
                color='black', linewidth=1.5, label='종가(close)')

        # ── MA5 선 그리기 ─────────────────────────────────────────
        # linestyle='--' : 점선 (실선은 '-', 점선은 '--')
        # alpha=0.8      : 투명도 (0=완전투명, 1=불투명, 0.8=약간 투명)
        #                  종가선이 더 잘 보이도록 MA5를 살짝 투명하게
        ax.plot(tdf['date'], tdf['ma5'],
                color='red', linewidth=1.2,
                linestyle='--', alpha=0.8,
                label='MA5 (5일 이동평균)')

        # ── 제목 및 축 라벨 ───────────────────────────────────────
        ax.set_title(ticker, fontsize=13, fontweight='bold')
        ax.set_xlabel('날짜', fontsize=10)
        if i == 0:
            # 첫 번째 칸에만 y축 라벨 표시 (나머지에 달면 중복됨)
            ax.set_ylabel('가격 (원)', fontsize=10)

        # ── x축 날짜 포맷 설정 ────────────────────────────────────
        # DateFormatter('%m-%d') : '2026-01-05' → '01-05' 형식으로 표시
        # %Y=연도 / %m=월 / %d=일
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

        # 날짜 눈금 개수를 최대 6개로 제한 (너무 빽빽하지 않게)
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=6))

        # x축 날짜 텍스트를 45도 기울이기 (겹쳐서 안 보이는 문제 해결)
        # ha='right' : 오른쪽 정렬 (기울였을 때 위치가 자연스럽게 맞춰짐)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)

        # ── y축 숫자 포맷 ─────────────────────────────────────────
        # 89250000 같은 큰 숫자에 쉼표 삽입: 89,250,000
        # FuncFormatter(lambda x, _: ...)  : 숫자(x)를 받아 포맷된 문자열 반환
        # f'{x:,.0f}' : 쉼표 포함, 소수점 없이
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
        )

        # ── 범례 표시 ─────────────────────────────────────────────
        # loc='best' : matplotlib이 차트 요소와 겹치지 않는 위치를 자동으로 찾아줌
        ax.legend(loc='best', fontsize=9)

        # ── 배경 격자 추가 ────────────────────────────────────────
        # alpha=0.3 : 격자를 연하게 (차트 선이 더 잘 보이도록)
        ax.grid(True, alpha=0.3, linestyle='--')

    # ── 전체 제목 ─────────────────────────────────────────────────
    # suptitle() : 서브플롯 전체를 감싸는 제목
    # y=1.02     : 제목을 그림 위쪽으로 살짝 올려서 겹침 방지
    fig.suptitle('암호화폐 종가 및 5일 이동평균선 (최근 30일)',
                 fontsize=15, fontweight='bold', y=1.02)

    # tight_layout() : 서브플롯 간 여백을 자동으로 조절해서 겹침 방지
    plt.tight_layout()
    plt.show()

    print("차트 출력 완료!")

In [ ]:
# ── 시각화 실행 ───────────────────────────────────────────────────

# 1단계: 종목별로 데이터 분리
ticker_data = prepare_plot_data(df)

# 2단계: 차트 출력
plot_close_and_ma5(ticker_data)

---
## Day 2 완료 체크리스트

- [ ] **Step 1** (01_data_collection): 3종목 30일치 수집 완료
- [ ] **Step 2** (02_preprocessing): `price_change`, `price_change_pct`, `high_low_diff` 컬럼 추가
- [ ] **Step 3** (03_moving_average): `ma5` 컬럼 추가, NaN 없음 확인
- [ ] **Step 4** (04_visualization): 종가 + MA5 서브플롯 차트 정상 출력

---

## Day 3 예고

오늘 만든 `ohlcv_final.csv`를  
**SQLite 데이터베이스에 저장**하고, SQL로 조회하는 방법을 배웁니다.  
데이터를 메모리(RAM)가 아닌 파일로 영구 저장하는 방법입니다!